# Perceiver CPI — Step C

Two questions, run as four sessions. Set `SESSION` in the next cell, run it, then come back and set the next letter.

| Session | What it runs | Roughly |
|---|---|---|
| `A` | Davis: the paper's protein encoder **plus** ESM-2, 2 seeds | 5–6 h |
| `B` | Metz: 400 epochs instead of 150, 2 seeds | 7–8 h |
| `C` | KIBA: 200 epochs instead of 150, seed 0 | 6–7 h |
| `D` | KIBA: the same, seed 1 | 6–7 h |
| **`E`** | **Davis: folds 1–4 of the 5-fold cross-validation** | **~5.5 h** |
| **`F`** | **Metz: folds 1–4 of the 5-fold cross-validation** | **~5 h** |

**Sessions E and F can be run at the same time** in two notebooks — they share no files. Together
with fold 0 (already trained in Phase 4) they give the 5-fold mean ± std that the paper reports,
so our numbers can finally be compared with its numbers in the same format.

**Session A** answers the protein question. Step B showed ESM-2 made the model read the protein for the first time (ranking 0.00 → +0.05) but cost accuracy (test MSE 0.643 → 0.734). Session A keeps both encoders, with the ESM part behind a gate that starts at zero — so the model begins as an exact copy of the paper's and learns how much pretrained protein knowledge to add.

**Sessions B, C and D** answer a different question and need no new code. Metz and KIBA sit far above what a model could reach *ignoring proteins altogether* (Metz 0.719 vs 0.332, KIBA 0.575 vs 0.270). That is under-training, so they get more epochs and a second seed to average.

### Before you run
1. **Add Input** (right panel) → **Datasets** → add your `perceiver cpi data` dataset.
2. **Settings → Accelerator → GPU T4 x2**, **Persistence → Files only**.
3. Set `SESSION` below.
4. **Save Version → Save & Run All (Commit)**, then close the browser.

### Carrying earlier work across
From session B onwards, also **Add Input → Your Work → Notebooks** and attach the previous session's output. Finished runs are then skipped, and the results tables at the end cover everything so far. Cell 1 prints how many runs it carried over, so you can see at a glance whether it worked.

In [ ]:
# 1. Which session to run, and find the data
SESSION = 'A'      # <-- A, then B, then C, then D

import os, shutil, subprocess, sys
from pathlib import Path

WHAT = {'A': ['--group', 'both'],     'B': ['--group', 'metz'],
        'C': ['kiba_long_s0'],       'D': ['kiba_long_s1'],
        'E': ['--group', 'davis_cv'], 'F': ['--group', 'metz_cv']}
if SESSION not in WHAT:
    raise ValueError(f'SESSION must be one of {sorted(WHAT)}, not {SESSION!r}')
NEEDS_ESM_BANK = SESSION == 'A'

INPUT = Path(os.environ.get('KAGGLE_INPUT', '/kaggle/input'))
WORK = Path(os.environ.get('KAGGLE_WORKING', '/kaggle/working'))

def find_source():
    for depth in range(1, 6):
        found = sorted(INPUT.glob('/'.join(['*'] * depth) + '/run_experiment.py'))
        if found:
            return found[0].parent
    return None

SOURCE = find_source()
if SOURCE is None:
    listing = '\n'.join(f'  {p.relative_to(INPUT)}' for p in sorted(INPUT.rglob('*'))[:40]) or '  (nothing)'
    raise FileNotFoundError(
        'Could not find run_experiment.py in the attached data.\n'
        'Use "Add Input" (right panel) -> Datasets -> add the dataset you uploaded, then Run All again.\n'
        f'What is attached right now:\n{listing}')
print('Found the data at:', SOURCE)

CODE = WORK / 'perceiver'
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
for item in SOURCE.iterdir():
    (shutil.copytree if item.is_dir() else shutil.copy2)(item, CODE / item.name)
print((CODE / 'BUNDLE_INFO.txt').read_text())

RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)
os.environ['PERCEIVER_RUNS'] = str(RUNS)

# Carry over earlier sessions' progress, if their output was attached as an input.
# Kaggle mounts inputs several folders deep, so search rather than assuming a depth.
carried = 0
for depth in range(1, 6):
    for previous in sorted(INPUT.glob('/'.join(['*'] * depth) + '/runs')):
        for folder in sorted(previous.iterdir()):
            if folder.is_dir() and not (RUNS / folder.name).exists():
                shutil.copytree(folder, RUNS / folder.name)
                print('Carried over:', folder.name)
                carried += 1
    if carried:
        break
print(f'Carried over {carried} earlier run(s).')
print('\nSession', SESSION, '->', ' '.join(WHAT[SESSION]))

In [ ]:
# 2. Show the GPU and install what the model needs (2-3 minutes)
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then Run All again.')
print('CPU cores:', os.cpu_count())

packages = ['rdkit', 'tape-proteins', 'typed-argument-parser', 'tensorboardX', 'lifelines']
if NEEDS_ESM_BANK:
    packages.append('fair-esm')
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:], result.stderr[-3000:])
    raise RuntimeError('Installing packages failed (see above).')

print(subprocess.run([sys.executable, '-c',
                      'import torch; print("PyTorch", torch.__version__, "| GPU:",'
                      ' torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 3. Session A only: work out each protein's ESM-2 features once (a few minutes on the GPU).
#    ESM is frozen, so its output never changes - recomputing it every epoch would waste the run.
if NEEDS_ESM_BANK:
    bank = CODE / 'esm_bank_davis.npz'
    if bank.exists():
        print('bank already present:', round(bank.stat().st_size / 1024**2, 1), 'MB')
    else:
        build = subprocess.run([sys.executable, 'make_esm_bank.py', 'davis'],
                               cwd=CODE, capture_output=True, text=True)
        print(build.stdout[-2500:], build.stderr[-2000:])
        if build.returncode != 0 or not bank.exists():
            raise RuntimeError('Building the ESM features failed (see above).')
else:
    print('This session uses the original protein encoder; no ESM features needed.')

In [ ]:
# 4. Quick check: one epoch of this session's first run (a few minutes)
check = subprocess.run([sys.executable, 'run_experiment.py', *WHAT[SESSION], '--epochs', '1'],
                       cwd=CODE, capture_output=True, text=True,
                       env={**os.environ, 'PERCEIVER_RUNS': str(WORK / 'check')})
print(check.stdout[-2500:], check.stderr[-2500:])
shutil.rmtree(WORK / 'check', ignore_errors=True)
if check.returncode != 0:
    raise RuntimeError('The check failed (see above). Nothing was trained, so no GPU time was lost.')
print('\nCheck passed.')

In [ ]:
# 5. Train this session. A progress line appears about once a minute.
os.chdir(CODE)
get_ipython().system(f'"{sys.executable}" run_experiment.py {" ".join(WHAT[SESSION])} {os.environ.get("PERCEIVER_TEST_ARGS", "")}')

In [ ]:
# 6. Results so far, across every run carried into this session.
for script in ('validation_preds.py', 'evaluate.py', 'protein_sensitivity.py', 'stepB_results.py'):
    if not (CODE / script).exists():
        continue
    out = subprocess.run([sys.executable, script], cwd=CODE, capture_output=True, text=True)
    print(f'----- {script} -----')
    print(out.stdout[-3000:] or out.stderr[-1500:])
for name in ('results.md', 'protein_sensitivity.md', 'stepB_results.md'):
    if (CODE / name).exists():
        shutil.copy2(CODE / name, WORK / name)

## When it finishes

Everything in `/kaggle/working` is saved as this notebook's output.

**Next session:** change `SESSION` to the next letter, attach this notebook's output as an input (Add Input → Your Work → Notebooks), and run again.

**After each one:** send Claude the output — the `kaggle kernels output ...` command at the bottom of the Output tab is the quickest way.

Two numbers matter, and they pull in opposite directions:

- **Protein ranking** — does the answer change correctly when you keep the drug and swap the protein? Everything before Step B scored 0.00. ESM-2 alone reached +0.05. The ceiling is about +0.41.
- **Test MSE** — plain accuracy. The paper gets 0.463 on Davis; our best so far is 0.622.

Session A is a success if it holds the accuracy of the original encoder *and* keeps the protein ranking that ESM-2 gained.